# Лабораторна робота №2. Частина 1
## Наука про дані: VHI-індекс для областей України

### Імпорт бібліотек

In [ ]:
import os
import re
import glob
import urllib.request
import pandas as pd
import numpy as np
from datetime import datetime
from io import StringIO

### Маппінг областей
В завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 — Cherkasy). Потрібно замінити індекси так, щоб області індексувалися за українською абеткою (1 — Вінницька).

In [ ]:
# NOAA province ID -> Ukrainian name
NOAA_TO_NAME = {
    1: "Черкаська", 2: "Чернігівська", 3: "Чернівецька",
    4: "АР Крим", 5: "Дніпропетровська", 6: "Донецька",
    7: "Івано-Франківська", 8: "Харківська", 9: "Херсонська",
    10: "Хмельницька", 11: "Київська", 12: "м. Київ",
    13: "Кіровоградська", 14: "Луганська", 15: "Львівська",
    16: "Миколаївська", 17: "Одеська", 18: "Полтавська",
    19: "Рівненська", 20: "м. Севастополь", 21: "Сумська",
    22: "Тернопільська", 23: "Закарпатська", 24: "Вінницька",
    25: "Волинська", 26: "Запорізька", 27: "Житомирська"
}

# Ukrainian alphabetical order: NOAA IDs sorted by Ukrainian name
UA_ORDER = [24, 25, 5, 6, 27, 23, 26, 7, 11, 13, 14, 15, 16, 17, 18, 19, 21, 22, 8, 9, 10, 1, 3, 2, 4, 12, 20]

# NOAA ID -> (Ukrainian index, Ukrainian name)
NOAA_TO_UA = {}
for ua_idx, noaa_id in enumerate(UA_ORDER, 1):
    NOAA_TO_UA[noaa_id] = (ua_idx, NOAA_TO_NAME[noaa_id])

print("Маппінг NOAA ID -> Український індекс:")
for noaa_id in sorted(NOAA_TO_UA.keys()):
    ua_idx, name = NOAA_TO_UA[noaa_id]
    print(f"  NOAA {noaa_id:2d} -> UA {ua_idx:2d}: {name}")

### Завантаження даних VHI
Для кожної адміністративної одиниці завантажити (urllib) файли з VHI-індексом. До імені файлу додається дата та час завантаження. Передбачено механізм запобігання повторного завантаження.

In [ ]:
def download_vhi_data(data_dir="vhi_data"):
    """Завантажує VHI дані для всіх областей України з NOAA."""
    os.makedirs(data_dir, exist_ok=True)
    
    base_url = (
        "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/"
        "get_TS_admin.php?country=UKR&provinceID={pid}"
        "&year1=1981&year2=2024&type=Mean"
    )
    
    now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    for pid in range(1, 28):
        # Перевірка чи дані вже завантажені
        existing = glob.glob(os.path.join(data_dir, f"vhi_id_{pid}_*.csv"))
        if existing:
            print(f"[skip] Province {pid} ({NOAA_TO_NAME[pid]}): вже завантажено — {os.path.basename(existing[-1])}")
            continue
        
        url = base_url.format(pid=pid)
        filename = f"vhi_id_{pid}_{now}.csv"
        filepath = os.path.join(data_dir, filename)
        
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"[ok]   Province {pid} ({NOAA_TO_NAME[pid]}) -> {filename}")
        except Exception as e:
            print(f"[err]  Province {pid} ({NOAA_TO_NAME[pid]}): {e}")

download_vhi_data()

### Зчитування та очищення даних
Зчитати завантажені файли у pandas DataFrame. Data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст. Додати стовпчики з назвою та індексом області.

In [ ]:
def read_vhi_file(filepath, noaa_id):
    """Зчитує один VHI файл та повертає очищений DataFrame."""
    with open(filepath, "r") as f:
        text = f.read()
    
    # Видалити HTML-теги
    text = re.sub(r"<[^>]+>", "", text)
    
    # Знайти початок даних (рядок з 'year')
    lines = text.strip().split("\n")
    header_idx = None
    for i, line in enumerate(lines):
        if re.match(r"\s*year\s+week", line, re.IGNORECASE):
            header_idx = i
            break
    
    if header_idx is None:
        print(f"  Warning: header not found in {filepath}")
        return pd.DataFrame()
    
    # Зібрати дані від заголовка
    data_lines = [lines[header_idx]]
    for line in lines[header_idx + 1:]:
        stripped = line.strip()
        if stripped and not stripped.startswith("<") and re.match(r"\s*\d", stripped):
            data_lines.append(line)
    
    data_text = "\n".join(data_lines)
    
    try:
        df = pd.read_csv(StringIO(data_text), sep=r"\s+", na_values=["-1", "-1.00"])
    except Exception as e:
        print(f"  Error parsing {filepath}: {e}")
        return pd.DataFrame()
    
    # Видалити зайві стовпці
    df = df.drop(columns=["SMN", "SMT"], errors="ignore")
    
    # Привести типи
    for col in ["year", "week"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ["VCI", "TCI", "VHI"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Видалити рядки без року/тижня
    df = df.dropna(subset=["year", "week"])
    df["year"] = df["year"].astype(int)
    df["week"] = df["week"].astype(int)
    
    # Додати інформацію про область
    ua_idx, ua_name = NOAA_TO_UA[noaa_id]
    df["noaa_id"] = noaa_id
    df["ua_index"] = ua_idx
    df["oblast"] = ua_name
    
    return df


def load_all_vhi(data_dir="vhi_data"):
    """Зчитує всі VHI файли та об'єднує в один DataFrame."""
    frames = []
    for pid in range(1, 28):
        files = sorted(glob.glob(os.path.join(data_dir, f"vhi_id_{pid}_*.csv")))
        if not files:
            print(f"  Warning: no file for province {pid}")
            continue
        filepath = files[-1]
        df = read_vhi_file(filepath, pid)
        if not df.empty:
            frames.append(df)
    
    if not frames:
        raise FileNotFoundError("Не знайдено жодного файлу з VHI даними!")
    
    result = pd.concat(frames, ignore_index=True)
    result = result.sort_values(["ua_index", "year", "week"]).reset_index(drop=True)
    
    # Заповнити пропуски інтерполяцією по області
    for col in ["VCI", "TCI", "VHI"]:
        result[col] = result.groupby("ua_index")[col].transform(
            lambda s: s.interpolate(method="linear", limit_direction="both")
        )
    
    return result


df_vhi = load_all_vhi()
print(f"\nЗагальний розмір: {df_vhi.shape}")
print(f"Області: {df_vhi['oblast'].nunique()}")
print(f"Роки: {df_vhi['year'].min()} — {df_vhi['year'].max()}")
df_vhi.head(10)

### Перевірка переіндексації
Області тепер впорядковані за українською абеткою.

In [ ]:
regions = df_vhi[["ua_index", "oblast"]].drop_duplicates().sort_values("ua_index")
print("Області за українською абеткою:")
for _, row in regions.iterrows():
    print(f"  {row['ua_index']:2d}. {row['oblast']}")

### Ряд VHI для області за вказаний рік
Функція повертає тижневий ряд VHI для обраної області та року.

In [ ]:
def vhi_for_region_year(df, ua_index, year):
    """Повертає ряд VHI для вказаної області (за укр. індексом) та року."""
    mask = (df["ua_index"] == ua_index) & (df["year"] == year)
    result = df.loc[mask, ["year", "week", "VCI", "TCI", "VHI", "oblast"]].copy()
    if result.empty:
        print(f"Дані не знайдено для області {ua_index}, рік {year}")
    return result

# Приклад: Вінницька область (1), 2020 рік
result = vhi_for_region_year(df_vhi, 1, 2020)
print(f"VHI для Вінницької області, 2020 рік ({len(result)} тижнів):")
result

### Ряд VHI за вказаний діапазон років для вказаних областей

In [ ]:
def vhi_for_regions_years(df, ua_indices, year_start, year_end):
    """Повертає VHI для вказаних областей та діапазону років."""
    mask = (
        df["ua_index"].isin(ua_indices) &
        (df["year"] >= year_start) &
        (df["year"] <= year_end)
    )
    result = df.loc[mask, ["year", "week", "VCI", "TCI", "VHI", "ua_index", "oblast"]].copy()
    if result.empty:
        print(f"Дані не знайдено для областей {ua_indices}, роки {year_start}-{year_end}")
    return result

# Приклад: Вінницька (1) та Київська (9), 2018-2020
result = vhi_for_regions_years(df_vhi, [1, 9], 2018, 2020)
print(f"Знайдено {len(result)} записів:")
result.head(10)

### Пошук екстремумів (min, max), середнього та медіани VHI
Для вказаних областей та діапазону років.

In [ ]:
def vhi_extremes(df, ua_indices, year_start, year_end):
    """Знаходить min, max, mean, median VHI для вказаних областей та років."""
    subset = vhi_for_regions_years(df, ua_indices, year_start, year_end)
    if subset.empty:
        return pd.DataFrame()
    
    stats = subset.groupby("oblast")["VHI"].agg(
        min="min", max="max", mean="mean", median="median"
    ).round(2)
    
    return stats

# Приклад: кілька областей, 2015-2023
regions = [1, 3, 9, 19]  # Вінницька, Дніпропетровська, Київська, Харківська
stats = vhi_extremes(df_vhi, regions, 2015, 2023)
print("Статистика VHI (2015-2023):")
stats

### Додаткові вибірки
Пошук тижнів з екстремальною посухою (VHI < 15) та сприятливих умов (VHI > 80).

In [ ]:
def vhi_drought_weeks(df, threshold=15):
    """Знаходить тижні з екстремальною посухою (VHI < threshold)."""
    mask = df["VHI"] < threshold
    result = df.loc[mask].copy()
    return result.sort_values("VHI")

def vhi_favorable_weeks(df, threshold=80):
    """Знаходить тижні зі сприятливими умовами (VHI > threshold)."""
    mask = df["VHI"] > threshold
    result = df.loc[mask].copy()
    return result.sort_values("VHI", ascending=False)

drought = vhi_drought_weeks(df_vhi)
print(f"Тижнів з екстремальною посухою (VHI < 15): {len(drought)}")
print("\nНайгірші 10 тижнів:")
drought[["year", "week", "oblast", "VHI"]].head(10)

In [ ]:
favorable = vhi_favorable_weeks(df_vhi)
print(f"Тижнів зі сприятливими умовами (VHI > 80): {len(favorable)}")
print("\nНайкращі 10 тижнів:")
favorable[["year", "week", "oblast", "VHI"]].head(10)